In [3]:
import sys

from brian2 import Hz
import numpy as np 
import pandas as pd 
import os 
import pickle

from brian2 import Hz,mV
from pathlib import Path
REPO_ROOT = Path.cwd().resolve().parents[2]
sys.path.insert(0, str(REPO_ROOT / "code"))
from model_unified_ver import run_exp
from model_unified_ver import default_params as params
import utils as utl
from analysis_of_simulation_result import *
from fitting import hill
from make_network import *
av1a1 = [720575940623041549,720575940622894616,720575940626958878,720575940633984924,720575940611137742,720575940627192337]


labial_cluster = pd.read_parquet('/volume_4/research/seongbong/flywire/geosmin_project_version_update/data/labial_cluster_info_v783.parquet')


In [6]:

params_opt_labial = pickle.load(open(REPO_ROOT/'figure4/figure4F-K.MN9_PER_fitting/fitting_result/labial/result_av1a1_170.pkl','rb'))

V_l1l2,K_l1l2,n_l1l2,V_l3,K_l3,n_l3,k_act_L,r0_L,h_L = params_opt_labial.x

firing = [(0,0),*[(np.round(hill(c,V_l1l2,K_l1l2,n_l1l2),1),np.round(hill(c,V_l3,K_l3,n_l3),1)) for c in [10,50,100,500]]]

print(firing)


[(0, 0), (9.1, 4.3), (19.9, 15.7), (25.7, 23.5), (37.9, 39.2)]


In [3]:
path = '/volume_4/research/seongbong/flywire/geosmin_project_version_update/figure4/PER_fitting/Seperate_fitting_subtype_all_at_once/convert_simulation_result/result_L1L2&L3'

total_paths = [f'{path}/{ff1}Hz_{ff2}Hz_170Hz.parquet' for ff1,ff2 in firing]
# interest_ids = inhibitory['atGRN+TPN1']+inhibitory['L1+L2+L3']#list(target_ids_valid_all[g_info_all==int(interest_g)])
rates,spikes,std = read_result_with_multi_thread(total_paths)

100%|██████████████████████████████████████████| 5/5 [00:00<00:00, 18110.12it/s]


In [4]:
inhibitory_neurons_concentration = []
for rate in rates:
    inhibitory_neurons_concentration.append([c for c in rate.index if np.sum(syn_df[syn_df.Presynaptic_ID==c]['Excitatory']<0)>0])#cell_in_network_per_type['av1a1'])


In [6]:
numiter = 0 
for ff1,ff2 in firing:
    #
    interest_neurons = inhibitory_neurons_concentration[numiter]
    interest_neurons_2_id = {a:b for b,a in enumerate(interest_neurons)}
    #
    this_spike_df = spikes[numiter][np.isin(spikes[numiter].flywire_id,interest_neurons)]
    spike_data_per_trial = []
    for i,t_df in this_spike_df.groupby(by='trial'):
        spike_data = {}
        temp = {}
        for j,spike_t_df in t_df.groupby(by='flywire_id'):
            temp[interest_neurons_2_id[j]] = spike_t_df.t.values
        for ii in range(len(interest_neurons)):
            if ii not in temp.keys():
                temp[ii] = np.array([])
        temp = dict(sorted(temp.items()))
        spike_data[1] = []
        spike_data[2] = []
        spike_data[3] = list(temp.values())
        spike_data_per_trial.append(spike_data)
    numiter += 1 
    with open(f'{os.getcwd()}/inhibitory_spike_data/{ff1}Hz_{ff2}Hz_170Hz.pkl','wb') as f:
        pickle.dump(spike_data_per_trial,f)
    

In [19]:
numiter = 0 
for ff1,ff2 in firing:
    params['w_syn'] = 0.275*mV
    params['n_run'] = 100
    config = {
        'path_res'  : f'{os.getcwd()}/result',                              # directory to store results
        'path_comp' : REPO_ROOT / "data" /'Completeness_783.csv',         # csv of the complete list of Flywire neurons
        'path_con'  : REPO_ROOT / "data" / 'Connectivity_783.parquet',   # connectivity data
        'n_proc'    : -1,                                               # number of CPU cores (-1: use all)
    }

    if 'input_spike_pattern' not in os.listdir(f'{os.getcwd()}/result'):
        os.mkdir(f"{config['path_res']}/input_spike_pattern")
    if 'params' not in os.listdir(f'{config["path_res"]}'):
        os.mkdir(f'{config["path_res"]}/params')

    pickle.dump(params,open(f'{config["path_res"]}/params/params.pkl','wb'))


    neu_exc = labial_c2g['L1']+labial_c2g['L2']
    
    interest_neurons = inhibitory_neurons_concentration[numiter]
    neu_add = [labial_c2g['L3'],interest_neurons]
    except_ids = [interest_neurons]
    except_wsyn = [0*mV]
    neu_slnc = []
    
    spike_path = REPO_ROOT/f'figure4/concentration_mapped_simulation/result_L1L2&L3/input_spike_pattern/{ff1}Hz_{ff2}Hz_170Hz.pkl'
    d = pickle.load(open(spike_path,'rb'))

    av1a1_spike_path = f'{os.getcwd()}/inhibitory_spike_data/{ff1}Hz_{ff2}Hz_170Hz.pkl'
    df_av1a1 = pickle.load(open(av1a1_spike_path,'rb'))

    spike_df = [{1:d[i][1],2:d[i][2],3:df_av1a1[i][3]} for i in range(params['n_run'])]

    with open(f'{config["path_res"]}/input_spike_pattern/{ff1}Hz_{ff2}Hz_170Hz.pkl','wb') as f:
        pickle.dump(spike_df,f)

    spike_path = f'{config["path_res"]}/input_spike_pattern/{ff1}Hz_{ff2}Hz_170Hz.pkl'
    predetermined_input_or_not = [1,1,1]
    params['r_poi1'] = ff1 * Hz
    params['r_poi2'] = ff2 * Hz
    params['r_poi3'] = 170 * Hz
    run_exp(exp_name=f'{ff1}Hz_{ff2}Hz_170Hz', neu_exc=neu_exc,neu_exc_add=neu_add,Except_input_Ids=except_ids,Except_input_w_syn=except_wsyn,neu_slnc=neu_slnc,params=params,predetermined_input_or_not=predetermined_input_or_not,spike_path=spike_path, **config)
    numiter += 1 

>>> Experiment:     0Hz_0Hz_170Hz
    Output file:    /volume_4/research/seongbong/flywire/geosmin_project_version_update/figure5/disinhibition_all_inhibitory/labial/result/0Hz_0Hz_170Hz.parquet
    Excited neurons: 428
    Elapsed time:   59 s
>>> Experiment:     9.1Hz_4.3Hz_170Hz
    Output file:    /volume_4/research/seongbong/flywire/geosmin_project_version_update/figure5/disinhibition_all_inhibitory/labial/result/9.1Hz_4.3Hz_170Hz.parquet
    Excited neurons: 511
    Elapsed time:   48 s
>>> Experiment:     19.9Hz_15.7Hz_170Hz
    Output file:    /volume_4/research/seongbong/flywire/geosmin_project_version_update/figure5/disinhibition_all_inhibitory/labial/result/19.9Hz_15.7Hz_170Hz.parquet
    Excited neurons: 540
    Elapsed time:   49 s
>>> Experiment:     25.7Hz_23.5Hz_170Hz
    Output file:    /volume_4/research/seongbong/flywire/geosmin_project_version_update/figure5/disinhibition_all_inhibitory/labial/result/25.7Hz_23.5Hz_170Hz.parquet
    Excited neurons: 512
    Elapsed t

In [22]:
numiter = 0 
for ff1,ff2 in firing:
    params['w_syn'] = 0.275*mV
    params['n_run'] = 100
    config = {
        'path_res'  : f'{os.getcwd()}/result_control',                              # directory to store results
        'path_comp' : REPO_ROOT / "data" /'Completeness_783.csv',         # csv of the complete list of Flywire neurons
        'path_con'  : REPO_ROOT / "data" / 'Connectivity_783.parquet',   # connectivity data
        'n_proc'    : -1,                                               # number of CPU cores (-1: use all)
    }


    if 'input_spike_pattern' not in os.listdir(f'{os.getcwd()}/result_control'):
        os.mkdir(f"{config['path_res']}/input_spike_pattern")
    if 'params' not in os.listdir(f'{config["path_res"]}'):
        os.mkdir(f'{config["path_res"]}/params')

    pickle.dump(params,open(f'{config["path_res"]}/params/params.pkl','wb'))


    neu_exc = labial_c2g['L1']+labial_c2g['L2']
    
    interest_neurons = inhibitory_neurons_concentration[0]
    neu_add = [labial_c2g['L3'],interest_neurons]
    except_ids = None
    except_wsyn = None
    neu_slnc = []
    
    spike_path = REPO_ROOT/f'figure4/concentration_mapped_simulation/result_L1L2&L3/input_spike_pattern/{ff1}Hz_{ff2}Hz_170Hz.pkl'
    d = pickle.load(open(spike_path,'rb'))

    av1a1_spike_path = f'{os.getcwd()}/inhibitory_spike_data/{0}Hz_{0}Hz_170Hz.pkl'
    df_av1a1 = pickle.load(open(av1a1_spike_path,'rb'))

    spike_df = [{1:d[i][1],2:d[i][2],3:df_av1a1[i][3]} for i in range(params['n_run'])]

    with open(f'{config["path_res"]}/input_spike_pattern/{ff1}Hz_{ff2}Hz_170Hz.pkl','wb') as f:
        pickle.dump(spike_df,f)

    spike_path = f'{config["path_res"]}/input_spike_pattern/{ff1}Hz_{ff2}Hz_170Hz.pkl'
    predetermined_input_or_not = [1,1,1]
    params['r_poi1'] = ff1 * Hz
    params['r_poi2'] = ff2 * Hz
    params['r_poi3'] = 170 * Hz
    run_exp(exp_name=f'{ff1}Hz_{ff2}Hz_170Hz', neu_exc=neu_exc,neu_exc_add=neu_add,Except_output_Ids=except_ids,Except_output_w_syn=except_wsyn,neu_slnc=neu_slnc,params=params,predetermined_input_or_not=predetermined_input_or_not,spike_path=spike_path, **config)
    numiter += 1 

>>> Skipping experiment 0Hz_0Hz_170Hz because /volume_4/research/seongbong/flywire/geosmin_project_version_update/figure5/disinhibition_all_inhibitory/labial/result_control/0Hz_0Hz_170Hz.parquet exists and force_overwrite = False
>>> Experiment:     9.1Hz_4.3Hz_170Hz
    Output file:    /volume_4/research/seongbong/flywire/geosmin_project_version_update/figure5/disinhibition_all_inhibitory/labial/result_control/9.1Hz_4.3Hz_170Hz.parquet
    Excited neurons: 428
    Elapsed time:   61 s
>>> Experiment:     19.9Hz_15.7Hz_170Hz
    Output file:    /volume_4/research/seongbong/flywire/geosmin_project_version_update/figure5/disinhibition_all_inhibitory/labial/result_control/19.9Hz_15.7Hz_170Hz.parquet
    Excited neurons: 428
    Elapsed time:   49 s
>>> Experiment:     25.7Hz_23.5Hz_170Hz
    Output file:    /volume_4/research/seongbong/flywire/geosmin_project_version_update/figure5/disinhibition_all_inhibitory/labial/result_control/25.7Hz_23.5Hz_170Hz.parquet
    Excited neurons: 428
    